In [ ]:
import os
os.chdir('/home/opokuml/kokoalert')

In [ ]:
import sys
sys.path.append('/home/opokuml/kokoalert')

from src.preprocess import file_to_spectrograms, check_recording_quality
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Test on one healthy clip
test_file = str(list(Path("data/raw/bowen/healthy").glob("*.wav"))[0])

# Quality check
quality = check_recording_quality(test_file)
print("Quality check:", quality)

# Get spectrograms
specs = file_to_spectrograms(test_file)
print(f"\nWindows produced: {len(specs)}")
print(f"Each window shape: {specs[0].shape}")  # Should be (128, 313, 1)

# Visualise the first window
plt.figure(figsize=(12, 4))
plt.imshow(specs[0][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='dB')
plt.title('Mel Spectrogram — Healthy Chicken Audio (Window 1)')
plt.xlabel('Time frames')
plt.ylabel('Mel frequency bands')
plt.tight_layout()
plt.show()

# Compare healthy vs sick
sick_file = str(list(Path("data/raw/bowen/sick").glob("*.wav"))[0])
sick_specs = file_to_spectrograms(sick_file)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(specs[0][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Healthy')
axes[0].set_xlabel('Time frames')
axes[0].set_ylabel('Mel bands')

axes[1].imshow(sick_specs[0][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('Sick')
axes[1].set_xlabel('Time frames')

plt.tight_layout()
plt.show()

In [ ]:
import os
os.chdir('/home/opokuml/kokoalert')

from src.preprocess import file_to_spectrograms, check_recording_quality
from pathlib import Path
import matplotlib.pyplot as plt

# Find first usable sick file
sick_files = list(Path("data/raw/bowen/sick").glob("*.wav"))
sick_specs = []
sick_file_used = None

for f in sick_files:
    specs = file_to_spectrograms(str(f))
    if len(specs) > 0:
        sick_specs = specs
        sick_file_used = f.name
        break

# Find first usable healthy file
healthy_files = list(Path("data/raw/bowen/healthy").glob("*.wav"))
healthy_specs = []
healthy_file_used = None

for f in healthy_files:
    specs = file_to_spectrograms(str(f))
    if len(specs) > 0:
        healthy_specs = specs
        healthy_file_used = f.name
        break

print(f"Healthy file: {healthy_file_used} — {len(healthy_specs)} windows")
print(f"Sick file: {sick_file_used} — {len(sick_specs)} windows")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].imshow(healthy_specs[0][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title(f'Healthy ({healthy_file_used})')
axes[0].set_xlabel('Time frames')
axes[0].set_ylabel('Mel bands')

axes[1].imshow(sick_specs[0][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title(f'Sick ({sick_file_used})')
axes[1].set_xlabel('Time frames')

plt.suptitle('Healthy vs Sick — Mel Spectrogram Comparison', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import os
os.chdir('/home/opokuml/kokoalert')

from src.preprocess import file_to_spectrograms
from pathlib import Path
import matplotlib.pyplot as plt

healthy_files = list(Path("data/raw/bowen/healthy").glob("*.wav"))
sick_files = list(Path("data/raw/bowen/sick").glob("*.wav"))

# Get 3 usable examples from each class
def get_usable(files, n=3):
    results = []
    for f in files:
        specs = file_to_spectrograms(str(f))
        if len(specs) > 0:
            results.append((f.name, specs[0]))
        if len(results) == n:
            break
    return results

healthy_samples = get_usable(healthy_files)
sick_samples = get_usable(sick_files)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, (name, spec) in enumerate(healthy_samples):
    axes[0][i].imshow(spec[:, :, 0], aspect='auto', origin='lower', cmap='viridis')
    axes[0][i].set_title(f'Healthy: {name}')
    axes[0][i].set_xlabel('Time frames')
    axes[0][i].set_ylabel('Mel bands')

for i, (name, spec) in enumerate(sick_samples):
    axes[1][i].imshow(spec[:, :, 0], aspect='auto', origin='lower', cmap='viridis')
    axes[1][i].set_title(f'Sick: {name}')
    axes[1][i].set_xlabel('Time frames')
    axes[1][i].set_ylabel('Mel bands')

plt.suptitle('3 Healthy vs 3 Sick — Can we see a difference?', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import os
os.chdir('/home/opokuml/kokoalert')

from src.anomaly_detector import build_autoencoder, compile_autoencoder

model = build_autoencoder()
model = compile_autoencoder(model)
model.summary()

# Working on the rish engine.

## Making exploration on the risk engine.

In [ ]:
import os
os.chdir('/home/opokuml/kokoalert')

from src.risk_engine import compute_farm_risk_score, format_risk_for_whatsapp
from datetime import datetime

# Test Case 1 — Worst case farmer
worst_case = {
    "region": "Ashanti",
    "flock_size": "over_2000",
    "vaccinated": False,
    "days_since_vaccination": 365,
    "has_footbath": False,
    "ventilation": "poor",
    "recent_deaths": True,
    "death_count_this_week": 5,
    "new_birds_introduced": True,
    "current_month": 11  # November — peak month
}

# Test Case 2 — Best case farmer
best_case = {
    "region": "Ashanti",
    "flock_size": "100_to_500",
    "vaccinated": True,
    "days_since_vaccination": 15,
    "has_footbath": True,
    "ventilation": "good",
    "recent_deaths": False,
    "new_birds_introduced": False,
    "current_month": 3  # March — lowest risk month
}

# Test Case 3 — Typical Ghanaian farmer
# Based on Ayim-Akonor et al. (2020) averages
typical = {
    "region": "Ashanti",
    "flock_size": "100_to_500",
    "vaccinated": True,
    "days_since_vaccination": 95,  # slightly overdue
    "has_footbath": False,          # 71.1% have no footbath
    "ventilation": "medium",
    "recent_deaths": False,
    "new_birds_introduced": False,
    "current_month": datetime.now().month
}

for name, profile in [("Worst case", worst_case),
                       ("Best case", best_case),
                       ("Typical farmer", typical)]:
    result = compute_farm_risk_score(profile)
    print(f"\n{'='*50}")
    print(f"{name}: {result['score']}/100 — {result['emoji']} {result['category']}")
    print(f"Risk factors: {len(result['risk_factors'])}")
    for rf in result['risk_factors']:
        print(f"  +{rf['points']} — {rf['factor']}")
    print(f"Top action: {result['top_action'][:80]}...")

print("\n\nWhatsApp format for typical farmer:")
print(format_risk_for_whatsapp(compute_farm_risk_score(typical)))

## Checking pipeline.py

In [ ]:
import os
os.chdir('/home/opokuml/kokoalert')

from src.pipeline import KokoAlertPipeline

pipeline = KokoAlertPipeline()
pipeline.load_models()

typical_farm = {
    "region": "Ashanti",
    "flock_size": "100_to_500",
    "vaccinated": True,
    "days_since_vaccination": 95,
    "has_footbath": False,
    "ventilation": "medium",
    "recent_deaths": False,
    "new_birds_introduced": False,
}

for label, path in [
    ("HEALTHY", "data/raw/bowen/healthy/3.wav"),
    ("SICK",    "data/raw/bowen/sick/5.wav")
]:
    print(f"\n{'='*50}")
    print(f"Testing: {label}")
    result = pipeline.analyse_audio(path, farm_profile=typical_farm)
    
    print(f"Status:          {result['status']}")
    print(f"Headline:        {result['headline']}")
    print(f"Windows:         {result.get('windows_analysed', 'N/A')}")
    print(f"Sick windows:    {result.get('sick_window_count', 'N/A')}")
    print(f"Avg probability: {result.get('avg_probability', 'N/A'):.3f}")
    
    if result['status'] == 'anomaly_detected':
        print(f"Risk score:      {result['risk_score']}/100 — {result['risk_category']}")
        print("Context lines:")
        for line in result['context_lines']:
            print(f"  {line}")

## Testing Biosecurity scorer script

In [1]:
import os
os.chdir('/home/opokuml/kokoalert')

from src.biosecurity_scorer import compute_biosecurity_score, format_biosecurity_for_whatsapp

worst = {
    "has_footbath": False,
    "ventilation": "poor",
    "new_birds_introduced": True,
    "flock_size": "over_2000"
}

best = {
    "has_footbath": True,
    "ventilation": "good",
    "new_birds_introduced": False,
    "flock_size": "100_to_500"
}

typical = {
    "has_footbath": False,
    "ventilation": "medium",
    "new_birds_introduced": False,
    "flock_size": "100_to_500"
}

for name, profile in [("Worst", worst), ("Best", best), ("Typical", typical)]:
    result = compute_biosecurity_score(profile)
    print(f"\n── {name} ──")
    print(f"Score: {result['score']}/10 — {result['grade']} {result['emoji']}")
    for imp in result['improvements']:
        print(f"  -{imp['points_lost']}pts — {imp['factor']}")

print("\n── WhatsApp (typical) ──")
print(format_biosecurity_for_whatsapp(compute_biosecurity_score(typical)))


── Worst ──
Score: 2/10 — Poor 🔴
  -3pts — New birds introduced without quarantine period
  -2pts — No footbath at poultry house entrance
  -2pts — Poor ventilation
  -1pts — Large flock in single house increases transmission risk

── Best ──
Score: 10/10 — Good 🟢

── Typical ──
Score: 7/10 — Fair 🟡
  -2pts — No footbath at poultry house entrance
  -1pts — Ventilation could be improved

── WhatsApp (typical) ──
🟡 *Biosecurity Score: 7/10 — Fair*

*Improvements needed:*
1. Install a footbath with 2% Virkon S or formalin solution at your poultry house entrance. Change solution every 3 days. 71.1% of Ashanti farms lack this — it is the easiest disease entry point to close.

2. Ensure all existing vents are fully open during the day. Consider adding roof ventilation if birds are above optimal stocking density.

_Reply RISK to see your disease risk score._


## Testing Vacination Tracker script

In [2]:
import os
os.chdir('/home/opokuml/kokoalert')

from src.vaccination_tracker import (
    get_all_vaccination_statuses,
    format_vaccination_for_whatsapp
)

overdue_farm = {
    "days_since_vaccination": 120,
    "flock_size": "100_to_500"
}

fresh_farm = {
    "days_since_vaccination": 15,
    "flock_size": "100_to_500"
}

upcoming_farm = {
    "days_since_vaccination": 65,
    "flock_size": "100_to_500"
}

for name, profile in [
    ("Overdue farm", overdue_farm),
    ("Fresh vaccination", fresh_farm),
    ("Coming up soon", upcoming_farm)
]:
    statuses = get_all_vaccination_statuses(profile)
    print(f"\n── {name} ──")
    for s in statuses:
        print(f"  {s['emoji']} {s['disease']}: {s['status_text']}")

print("\n── WhatsApp output (overdue farm) ──")
print(format_vaccination_for_whatsapp(
    get_all_vaccination_statuses(overdue_farm)
))


── Overdue farm ──
  ❗ Newcastle Disease: OVERDUE by 30 days
  ❗ Infectious Bursal Disease (Gumboro): OVERDUE by 99 days
  ❗ Infectious Bronchitis: OVERDUE by 92 days

── Fresh vaccination ──
  ❗ Infectious Bursal Disease (Gumboro): Due in 6 days
  🟡 Infectious Bronchitis: Due in 13 days
  ✅ Newcastle Disease: Due in 75 days

── Coming up soon ──
  ❗ Infectious Bursal Disease (Gumboro): OVERDUE by 44 days
  ❗ Infectious Bronchitis: OVERDUE by 37 days
  🟡 Newcastle Disease: Due in 25 days

── WhatsApp output (overdue farm) ──
💉 *Vaccination Schedule*

❗ *Newcastle Disease*
Status: OVERDUE by 30 days
Next due: 06 May 2026
Vaccine: I-2 Thermostable
⚠️ 70–100% in unvaccinated flocks

❗ *Infectious Bursal Disease (Gumboro)*
Status: OVERDUE by 99 days
Next due: 06 May 2026
Vaccine: Bursine-2
⚠️ Up to 30% mortality, high morbidity

❗ *Infectious Bronchitis*
Status: OVERDUE by 92 days
Next due: 06 May 2026
Vaccine: H120 strain
⚠️ Low mortality, high production losses

_Reply RISK to see your 